In [ ]:
# Analisis exploratorio del test GAIT Arms

# Este cuaderno documenta los pasos para cargar y revisar los datos IMU registrados en BASE-SPINE, LEFT-HAND y RIGHT-HAND durante el test GAIT_ARMS.

In [2]:
# Librerias y ubicacion del archivo
%pip install plotly

import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

DATA_PATH = Path("../../data/20250929214532_19305147-961dbd9b-3093-4864-9daa-b9256b66691d.json")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontro el archivo esperado en {DATA_PATH}")

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Carga y normalizacion del JSON
with DATA_PATH.open(encoding="utf-8") as f:
    raw = json.load(f)

patient = raw.get("patient", {})
test_type = raw.get("testType")

records = []
for entry in raw.get("imuData", []):
    device = entry.get("deviceId")
    ts = entry.get("timestamp")
    acc = entry.get("accelerometer", {})
    gyro = entry.get("gyroscope", {})
    records.append(
        {
            "device": device,
            "timestamp": ts,
            "acc_x": acc.get("x"),
            "acc_y": acc.get("y"),
            "acc_z": acc.get("z"),
            "gyro_x": gyro.get("x"),
            "gyro_y": gyro.get("y"),
            "gyro_z": gyro.get("z"),
        }
    )

imu_df = pd.DataFrame.from_records(records)
if imu_df.empty:
    raise ValueError("No se encontraron registros en imuData")
imu_df.sort_values(["device", "timestamp"], inplace=True)
imu_df.reset_index(drop=True, inplace=True)
imu_df["acc_mag"] = np.sqrt(imu_df["acc_x"] ** 2 + imu_df["acc_y"] ** 2 + imu_df["acc_z"] ** 2)
imu_df["gyro_mag"] = np.sqrt(imu_df["gyro_x"] ** 2 + imu_df["gyro_y"] ** 2 + imu_df["gyro_z"] ** 2)
imu_df.head()

,device,timestamp,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,acc_mag,gyro_mag
0,BASE-SPINE,780996,-0.073120,0.127930,0.987549,0.366211,-0.305176,-0.244141,0.998482,0.535582
1,BASE-SPINE,781022,-0.072266,0.127808,0.987671,0.305176,-0.305176,-0.244141,0.998525,0.495852
2,BASE-SPINE,781049,-0.072754,0.127441,0.987427,0.305176,-0.305176,-0.244141,0.998272,0.495852
3,BASE-SPINE,781075,-0.071655,0.128174,0.987671,0.366211,-0.305176,-0.244141,0.998527,0.535582
4,BASE-SPINE,781101,-0.072754,0.127441,0.987305,0.366211,-0.305176,-0.305176,0.998151,0.566017


In [4]:
# Resumen del paciente y conteos globales
print(f"Paciente: {patient.get('name', 'N/D')} ({patient.get('id', 'N/D')})")
print(f"Fecha de nacimiento: {patient.get('birthDate', 'N/D')}")
print(f"Observaciones: {patient.get('observations') or 'Sin observaciones registradas'}")
print(f"Tipo de prueba: {test_type}")
print(f"Total de registros IMU: {imu_df.shape[0]:,}")
print(f"Dispositivos disponibles: {sorted(imu_df['device'].unique().tolist())}")

Paciente: Luis Alberto Londoño (19305147)
Fecha de nacimiento: 02/04/1956
Observaciones: Sin observaciones registradas
Tipo de prueba: GAIT_ANKLES
Total de registros IMU: 2,100
Dispositivos disponibles: ['BASE-SPINE', 'LEFT-ANKLE', 'RIGHT-ANKLE']


In [5]:
# Brecha temporal y frecuencia de muestreo aproximada
def summarize_sampling(df: pd.DataFrame) -> pd.DataFrame:
    stats = []
    for device, grp in df.groupby("device"):
        timestamps = grp["timestamp"].astype(float).sort_values()
        deltas = timestamps.diff().dropna()
        stats.append(
            {
                "device": device,
                "samples": len(grp),
                "t_start": timestamps.iloc[0],
                "t_end": timestamps.iloc[-1],
                "duration_ms": timestamps.iloc[-1] - timestamps.iloc[0],
                "mean_dt": deltas.mean() if not deltas.empty else np.nan,
                "median_dt": deltas.median() if not deltas.empty else np.nan,
                "min_dt": deltas.min() if not deltas.empty else np.nan,
                "max_dt": deltas.max() if not deltas.empty else np.nan,
            }
        )
    return pd.DataFrame(stats)

sampling_summary = summarize_sampling(imu_df)
sampling_summary = sampling_summary.assign(
    duration_s=lambda d: d["duration_ms"] / 1000,
    mean_hz=lambda d: 1000 / d["mean_dt"].replace({0: np.nan}),
    median_hz=lambda d: 1000 / d["median_dt"].replace({0: np.nan}),
)
sampling_summary

,device,samples,t_start,t_end,duration_ms,mean_dt,median_dt,min_dt,max_dt,duration_s,mean_hz,median_hz
0,BASE-SPINE,756,780996.0,800860.0,19864.0,26.309934,26.0,26.0,27.0,19.864,38.008458,38.461538
1,LEFT-ANKLE,691,842374.0,860519.0,18145.0,26.297101,26.0,26.0,52.0,18.145,38.027005,38.461538
2,RIGHT-ANKLE,653,837938.0,856925.0,18987.0,29.121166,26.0,26.0,85.0,18.987,34.339285,38.461538


In [6]:
# Estadisticas descriptivas por dispositivo
metric_cols = [
    "acc_x",
    "acc_y",
    "acc_z",
    "acc_mag",
    "gyro_x",
    "gyro_y",
    "gyro_z",
    "gyro_mag",
]
desc = imu_df.groupby("device")[metric_cols].agg(["mean", "std", "min", "max"])
desc.columns = [f"{col}_{stat}" for col, stat in desc.columns]
desc.reset_index()

,device,acc_x_mean,acc_x_std,acc_x_min,acc_x_max,acc_y_mean,acc_y_std,acc_y_min,acc_y_max,acc_z_mean,...,gyro_y_min,gyro_y_max,gyro_z_mean,gyro_z_std,gyro_z_min,gyro_z_max,gyro_mag_mean,gyro_mag_std,gyro_mag_min,gyro_mag_max
0,BASE-SPINE,-0.072176,0.000491,-0.073486,-0.070068,0.127990,0.000570,0.125977,0.129639,0.987423,...,-0.427246,-0.12207,-0.246886,0.019677,-0.305176,-0.183105,0.507737,0.049199,0.299010,0.719593
1,LEFT-ANKLE,0.930203,0.445297,0.081055,3.239380,0.328833,0.376097,-1.370483,1.245605,-0.213797,...,-31.860350,37.17041,-0.473884,33.238660,-99.121090,111.084000,36.750444,33.602458,0.061035,144.708451
2,RIGHT-ANKLE,0.959197,0.008791,0.829346,1.066772,-0.186383,0.015041,-0.375122,-0.053467,-0.224518,...,-8.361816,5.92041,-0.201425,1.507179,-14.282230,5.432129,2.501455,1.878628,0.172633,16.847919


In [7]:
# Valores faltantes por dispositivo y variable
missing = (
    imu_df.drop(columns=["device"])
    .isna()
    .groupby(imu_df["device"])
    .sum()
    .reset_index()
    .rename(columns={"index": "device"})
)
missing

,device,timestamp,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,acc_mag,gyro_mag
0,BASE-SPINE,0,0,0,0,0,0,0,0,0
1,LEFT-ANKLE,0,0,0,0,0,0,0,0,0
2,RIGHT-ANKLE,0,0,0,0,0,0,0,0,0


In [8]:
# Serie temporal de la magnitud de aceleracion
fig_acc = px.line(
    imu_df,
    x="timestamp",
    y="acc_mag",
    color="device",
    title="Magnitud de aceleracion por dispositivo",
    labels={"timestamp": "timestamp (ms)", "acc_mag": "|a| (g)"},
)
fig_acc

In [9]:
# Serie temporal de la magnitud de velocidad angular
fig_gyro = px.line(
    imu_df,
    x="timestamp",
    y="gyro_mag",
    color="device",
    title="Magnitud de velocidad angular por dispositivo",
    labels={"timestamp": "timestamp (ms)", "gyro_mag": "|w| (deg/s)"},
)
fig_gyro

In [10]:
# Componentes individuales del acelerometro
acc_long = imu_df.melt(
    id_vars=["device", "timestamp"],
    value_vars=["acc_x", "acc_y", "acc_z"],
    var_name="axis",
    value_name="acc"
)
fig_acc_axis = px.line(
    acc_long,
    x="timestamp",
    y="acc",
    color="device",
    facet_row="axis",
    title="Componentes del acelerometro",
    labels={"acc": "aceleracion (g)", "timestamp": "timestamp (ms)"},
)
fig_acc_axis.update_layout(showlegend=True)
fig_acc_axis

In [11]:
# Componentes individuales del giroscopio
gyro_long = imu_df.melt(
    id_vars=["device", "timestamp"],
    value_vars=["gyro_x", "gyro_y", "gyro_z"],
    var_name="axis",
    value_name="gyro"
)
fig_gyro_axis = px.line(
    gyro_long,
    x="timestamp",
    y="gyro",
    color="device",
    facet_row="axis",
    title="Componentes del giroscopio",
    labels={"gyro": "velocidad angular (deg/s)", "timestamp": "timestamp (ms)"},
)
fig_gyro_axis.update_layout(showlegend=True)
fig_gyro_axis

## Notas y proximos pasos
- Revisar ventanas de interes en las señales crudas para detectar artefactos o segmentos relevantes dle test.
- Aplicar filtrado o suavizado adicional antes de extraer indicadores clinicos (por ejemplo, gait speed, balanceo de brazos).
- Comparar este registro con otras sesiones del paciente para evaluar consistencia y progresion.